# Insurance Claim Fraud Detection — EDA and Modeling

Run this notebook on the **lab computer** after placing the Kaggle CSV at `data/insurance_fraud_detection.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import RocCurveDisplay, confusion_matrix

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from preprocessing import build_model_pipeline, get_feature_columns, load_raw_data, split_data
from utils import OUTPUTS_DIR, TARGET_COLUMN

EDA_DIR = OUTPUTS_DIR / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

In [ ]:
df = load_raw_data()
print('Shape:', df.shape)
df.head()

In [ ]:
missing = df.replace('?', pd.NA).isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()
target_counts

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
target_counts.plot(kind='bar', ax=ax, color=['#2ca02c', '#d62728'])
ax.set_title('Fraud Reported Class Distribution')
ax.set_xlabel('fraud_reported')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(EDA_DIR / 'target_distribution.png', dpi=150)
plt.show()

In [ ]:
numeric_cols = [
    'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim',
    'months_as_customer', 'age', 'witnesses'
]
available = [col for col in numeric_cols if col in df.columns]
summary = df.groupby(TARGET_COLUMN)[available].mean().T
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x=TARGET_COLUMN, y='total_claim_amount', ax=ax)
ax.set_title('Total Claim Amount by Fraud Label')
plt.tight_layout()
plt.savefig(EDA_DIR / 'claim_amount_by_fraud.png', dpi=150)
plt.show()

In [ ]:
corr_df = df.replace('?', pd.NA)
for col in corr_df.columns:
    if corr_df[col].dtype == 'object':
        continue
    corr_df[col] = pd.to_numeric(corr_df[col], errors='coerce')
corr = corr_df.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Numeric Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(EDA_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
numeric_features, categorical_features = get_feature_columns(df)

models = {
    'logistic': build_model_pipeline(numeric_features, categorical_features, 'logistic', use_smote=False),
    'random_forest': build_model_pipeline(numeric_features, categorical_features, 'random_forest', use_smote=True),
}

comparison_rows = []
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    comparison_rows.append({
        'model': name,
        'accuracy': (y_pred == y_test).mean(),
        'precision': ((y_pred == 1) & (y_test == 1)).sum() / max((y_pred == 1).sum(), 1),
        'recall': ((y_pred == 1) & (y_test == 1)).sum() / max((y_test == 1).sum(), 1),
    })

comparison = pd.DataFrame(comparison_rows)
comparison

In [ ]:
best_model = models['random_forest']
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Random Forest Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(EDA_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title('Random Forest ROC Curve')
plt.tight_layout()
plt.savefig(EDA_DIR / 'roc_curve.png', dpi=150)
plt.show()